# Best Price Seeker

Here's an Agent that sources for the best deals for you on Jumia.

Uses our custom build fine-tunned model based off of DeepSeek's deepseek-llm-7b-base model.

In [3]:
%pip install -q feedparser

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[1]))

In [3]:
import chromadb
import gradio as gr
import logging
import json
import modal
import os
import requests

from tqdm.notebook import tqdm
from agents.preprocessor import Preprocessor
from dotenv import load_dotenv
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
from litellm import completion
from openai import OpenAI

from llama import app
from pricer_ephemeral import app, price
from agents.autonomous_planning_agent import AutonomousPlanningAgent
from agents.items import Item
from agents.evaluator import evaluate
from deals import ScrapedDeal, DealSelection, Opportunity, Deal
from agents.scanner_agent import ScannerAgent
from agents.messaging_agent import MessagingAgent
from deal_agent_framework import DealAgentFramework

In [4]:
load_dotenv(override=True)

root = logging.getLogger()
root.setLevel(logging.INFO)

DB = "products_vectorstore"

os.environ["PYTHONIOENCODING"] = "utf-8"
LITE_MODE = True

OSS_MODEL = "ollama/gpt-oss:20b"
OUR_TRAINED_MODEL = "seuun/price-deepseek-2026-03-09_11.24.11-lite"

openai = OpenAI(base_url="http://localhost:11434", api_key="ollama")

hf_token = os.environ['HF_TOKEN']

pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

### HuggingFace Login

In [5]:
login(token=hf_token, add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
username = "seuun"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [ ]:
preprocessor = Preprocessor(model_name="groq/openai/gpt-oss-20b")

def preprocess(text):
    text = preprocessor.preprocess(text)
    print(text)

In [10]:
!uv run modal deploy -m pricer_service
!modal deploy -m pricer_service2

Using CPython 3.12.12
Creating virtual environment at: /Users/seunaminu/Documents/trainings_python/llm_engineering_ed_donner/.venv
Installed 244 packages in 1.40s4.0.15                            ░░░░░░░░░░░░░░░░░░░░ [0/0] Installing wheels...                                 
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /Users/seunaminu/Documents/trainings_python/llm_engineering_ed_donner/.venv/ │
│ lib/python3.12/site-packages/modal/cli/import_refs.py:75 in                  │
│ import_file_or_module                                                        │
│                                                                              │
│    74 │   │   try:                                                           │
│ ❱  75 │   │   │   module = importlib.import_module(import_ref.file_or_module │
│    76 │   │   except Exception as exc:                                       │
│                                                                          

In [ ]:
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()

In [ ]:
def price(text):
    text = preprocess(text)
    reply = pricer.price.remote(text)
    print(reply)

## Initiate Database

In [ ]:
client = chromadb.PersistentClient(path=DB)

## Bringing in our SentenceTransformer Encoding LLM

In [ ]:
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
# Check if the collection exists; if not, create it

collection_name = "products"
existing_collection_names = [collection.name for collection in client.list_collections()]

if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    for i in tqdm(range(0, len(train), 1000)):
        documents = [item.summary for item in train[i: i+1000]]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
        ids = [f"doc_{j}" for j in range(i, i+1000)]
        ids = ids[:len(documents)]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

collection = client.get_or_create_collection(collection_name)

## Introducing our RAG Context

In [ ]:
def make_context(similars, prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

In [ ]:
def messages_for(item, similars, prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [ ]:
def vector(item):
    return encoder.encode(item.summary)

In [ ]:
def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)
    documents = results['documents'][0][:]
    prices = [m['price'] for m in results['metadatas'][0][:]]
    return documents, prices

In [ ]:
def trained_model_rag(item):
    documents, prices = find_similars(item)
    response = completion(model="ollama/gpt-oss:20b", messages=messages_for(item, documents, prices), reasoning_effort="high", seed=42)
    return response.choices[0].message.content

In [ ]:
evaluate(trained_model_rag, test)

# Deals Time!

In [ ]:
feeds = [
    "https://www.jumia.com.ng/phones-tablets/",
    "https://www.jumia.com.ng/electronics/",
    "https://www.jumia.com.ng/computing/"
]

In [ ]:
deals = ScrapedDeal.fetch(show_progress=True, feeds=feeds)

### We are going to ask GPT-OSS to summarize deals and identify their price

In [ ]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [ ]:
def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [ ]:
user_prompt = make_user_prompt(deals)
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

In [ ]:
response = openai.chat.completions.parse(model=OSS_MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed

In [ ]:
agent = ScannerAgent()
result = agent.scan()

## Introducing Pushover

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
agent = MessagingAgent()

## Tools Call

In [ ]:
scan_function = {
        "name": "scan_the_internet_for_bargains",
        "description": "Returns top bargains scraped from the internet along with the price each item is being offered for",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False
        }
    }

estimate_function = {
    "name": "estimate_true_value",
    "description": "Given the description of an item, estimate how much it is actually worth",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "The description of the item to be estimated"
            },
        },
        "required": ["description"],
        "additionalProperties": False
    }
}

notify_function = {
    "name": "notify_user_of_deal",
    "description": "Send the user a push notification about the single most compelling deal; only call this one time",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "The description of the item itself scraped from the internet"
            },
            "deal_price": {
                "type": "number",
                "description": "The price offered by this deal scraped from the internet"
            }
            ,
            "estimated_true_value": {
                "type": "number",
                "description": "The estimated actual value that this is worth"
            }
            ,
            "url": {
                "type": "string",
                "description": "The URL of this deal as scraped from the internet"
            }
        },
        "required": ["description", "deal_price", "estimated_true_value", "url"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [{"type": "function", "function": scan_function},
 {"type": "function", "function": estimate_function},
 {"type": "function", "function": notify_function}
]

In [ ]:
def handle_tool_call(message):
    """
    Actually call the tools associated with this message
    """
    results = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
system_message = "You find great deals on bargain products using your tools, and notify the user of the best bargain."
user_message = """
First, use your tool to scan the internet for bargain deals. Then for each deal, use your tool to estimate its true value.
Then pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to notify the user.
Then just reply OK to indicate success.
"""
messages = [{"role": "system", "content": system_message},{"role": "user", "content": user_message}]

# Finally, Our Agent 😎

In [ ]:
done = False
while not done:
    response = openai.chat.completions.create(model=OUR_TRAINED_MODEL, messages=messages, tools=tools)
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        results = handle_tool_call(message)
        messages.append(message)
        messages.extend(results)
    else:
        done = True
response.choices[0].message.content

In [ ]:
agent = AutonomousPlanningAgent(collection)

agent.plan()

# Interacting with our Agent

In [ ]:
agent_framework = DealAgentFramework()
agent_framework.init_agents_as_needed()

with gr.Blocks(title="The Price is Right", fill_width=True) as ui:

    initial_deal = Deal(product_description="Example description", price=100.0, url="https://cnn.com")
    initial_opportunity = Opportunity(deal=initial_deal, estimate=200.0, discount=100.0)
    opportunities = gr.State([initial_opportunity])

    def get_table(opps):
        return [[opp.deal.product_description, opp.deal.price, opp.estimate, opp.discount, opp.deal.url] for opp in opps]

    def do_select(opportunities, selected_index: gr.SelectData):
        row = selected_index.index[0]
        opportunity = opportunities[row]
        agent_framework.planner.messenger.alert(opportunity)

    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:24px">"The Price is Right" - Deal Hunting Agentic AI</div>')
    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:14px">Deals surfaced so far:</div>')
    with gr.Row():
        opportunities_dataframe = gr.Dataframe(
            headers=["Description", "Price", "Estimate", "Discount", "URL"],
            wrap=True,
            column_widths=[4, 1, 1, 1, 2],
            row_count=10,
            col_count=5,
            max_height=400,
        )

    ui.load(get_table, inputs=[opportunities], outputs=[opportunities_dataframe])
    opportunities_dataframe.select(do_select, inputs=[opportunities], outputs=[])

ui.launch(inbrowser=True)

# Publish!

In [ ]:
!uv run price_is_right.py